# 저장된 최적 모델(best_model.pkl) 불러오기 및 실전 평가

다른 PC(시험장)에서 기존에 학습 완료된 모델을 불러와 새로운 데이터(test)를 채점하는 전용 코드입니다.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.metrics import f1_score, classification_report

# 1. 모델 파일 확인 및 로드
model_path = 'best_model.pkl'

if not os.path.exists(model_path):
    print(f"❌ [오류] {model_path} 파일이 같은 폴더에 없습니다! 먼저 업로드/다운로드 해주세요.")
else:
    with open(model_path, 'rb') as f:
        model_data = pickle.load(f)
        
    model = model_data['model']
    selected_features = model_data['features']
    scaler = model_data['scaler']
    threshold = model_data.get('threshold', 0.5)
    
    print(f"✅ 학습된 모델 로드 성공! (알고리즘: {type(model).__name__})")
    print(f"📌 학습에 사용된 5개 피처: {selected_features}")
    print(f"📌 결정 임계값: {threshold:.3f}")


## 새로운 테스트 데이터(채점용 데이터) 불러오기 및 예측 수행

교수님이 새로 제공하신 CSV 파일의 이름을 아래 `test_csv_path` 변수에 넣어주세요.

In [ ]:
# 2. 실시간 테스트 데이터 평가 및 채점
test_csv_path = 'creditcard.csv' # <-- 교수님이 주신 새 파일명으로 변경하세요!

if not os.path.exists(test_csv_path):
    print(f"❌ [오류] 채점용 데이터 '{test_csv_path}' 파일이 없습니다. 이름을 확인해주세요.")
else:
    try:
        test_df = pd.read_csv(test_csv_path, encoding='utf-8')
    except UnicodeDecodeError:
        test_df = pd.read_csv(test_csv_path, encoding='cp949')
        
    print(f"✅ 테스트 데이터 로드 완료! shape: {test_df.shape}")
    
    # 3. 필수 피처 존재 여부 확인
    missing_features = [f for f in selected_features if f not in test_df.columns]
    if missing_features:
        print(f"❌ [오류] 테스트 데이터에 필수 피처 {missing_features}이 없습니다.")
    else:
        # 데이터 분리 및 스케일링
        X_test = test_df[selected_features]
        X_test_scaled = scaler.transform(X_test) if scaler is not None else X_test.values
        
        # 모델 예측
        probs = model.predict_proba(X_test_scaled)[:, 1]
        predictions = (probs >= threshold).astype(int)
        
        # 예측값 저장
        output_df = pd.DataFrame({'prediction': predictions})
        output_df.to_csv('predictions.csv', index=False)
        print(f"✅ 예측 완료! 결과가 'predictions.csv'로 저장되었습니다.")
        
        # 4. 정답(Class) 컬럼이 있을 경우 자동 채점 (교수님이 주신 데이터에 정답이 있을 때만 동작)
        if 'Class' in test_df.columns:
            y_test = test_df['Class']
            macro_f1 = f1_score(y_test, predictions, average='macro')
            
            print("\n" + "="*50)
            print("📊 기말고사 실시간 채점 결과 보고서 📊")
            print("="*50)
            print(classification_report(y_test, predictions))
            print(f"⭐ 최종 Macro F1-Score: {macro_f1:.5f} ⭐")
            print("="*50)
        else:
            print("\n💡 [알림] 정답(Class) 컬럼이 없어 채점은 생략하고 예측 파일만 생성했습니다.")
